# First doing the imports

In [106]:
from whoosh.index import create_in
from whoosh.fields import *
import json
import os
from tqdm import tqdm
from whoosh.query import *
from whoosh.qparser import QueryParser, FuzzyTermPlugin
import linecache

# Now we creat the index but we don't open it yet.

In [8]:
schema = Schema(doc_id = ID(stored=True), article_title = TEXT(stored=True), \
                section_title = TEXT(stored=True), content=TEXT(stored=False))

ix = create_in("woosh_index",schema)

# I want to look at the json chuncks first and perhaps do some prelinary cleanup getting the special sysmbols out of there.

In [9]:
path_to_doc_chuncks = '/Users/tylergarfield/Desktop/school/CSC483/Rags_To_Searches/corpus/statpearls/chunk'
docs_in_durr = os.listdir(path_to_doc_chuncks)

with ix.writer() as ind_writer:

    for file in tqdm(docs_in_durr,total=len(docs_in_durr)):
    
        with open(f'{path_to_doc_chuncks}/{file}','r',encoding="utf-8") as f:
            
            for line in f:
                
                curr_art_chunk = json.loads(line)

                doc_id_curr = curr_art_chunk['id']
                cont_curr = curr_art_chunk['content']

                article_sec = curr_art_chunk['title'].split('--')
                main_title_curr = article_sec[0]
                sec_header_curr = article_sec[1]

                ind_writer.add_document(doc_id=doc_id_curr,article_title=main_title_curr,\
                                        section_title=sec_header_curr,content=cont_curr)
    


100%|███████████████████████████████████████████████████████████████████████████████| 9632/9632 [01:04<00:00, 150.34it/s]


# Now that the index has been built I am going to test out querying it

In [113]:
def get_article_content(article_id):
    part_ids = article_id.split('_')
    
    main_article_id = part_ids[0]
    chunck_id = part_ids[1]

    line = linecache.getline(f'{path_to_doc_chuncks}/{main_article_id}.jsonl',(int(chunck_id)+1))
    art_json = json.loads(line)

    return art_json['content']

In [135]:
with ix.searcher() as searcher:
    parser = QueryParser("content",ix.schema)
    parser.add_plugin(FuzzyTermPlugin())

    #query_curr = parser.parse('section_title:(history AND physical)^10 AND ((headache AND backache)^0.2 OR diarhea~2^3.5 AND vomiting^2)')
    #query_curr = parser.parse('section_title:(history AND physical)^2 AND content:"headache vomiting"~1')
    query_curr = parser.parse('section_title:(history AND physical)^2 AND (content:headache~2^2 vomiting~2^2)')
    print(query_curr)
    print('\n\n')
    
    test_res = searcher.search(query_curr,limit=20)

    print("-------------Top Res Content------------")
    print(f'{test_res[0]["doc_id"]}: {get_article_content(test_res[0]["doc_id"])}\n\n')

    print("-------------Worst Res Content------------")
    print(f'{test_res[len(test_res.top_n)-1]["doc_id"]}: {get_article_content(test_res[len(test_res.top_n)-1]["doc_id"])}\n\n')


    print("-------------Possible Diseases------------")
    for res in test_res:
        print(f'\t {res.rank}) {res["article_title"]} |{res.score:.4f}|')

(section_title:history^2.0 AND section_title:physical^2.0 AND content:headache~2^4.000000 AND content:vomiting~2^4.000000)



-------------Top Res Content------------
article-27743_23: The location of the tumor, supratentorial or infratentorial, is important for the manifestation of symptoms. Those with an infratentorial location usually develop hydrocephalus with headache, vomiting, irritability, and lethargy. Ataxia or other cerebellar signs and cranial nerve palsies are common. They rarely have seizures. In supratentorial locations, vomiting, seizures, and headaches are common. Hemiparesis is present if the tumor affects the cortical motor areas or the descending tracts. The presentation is also affected by age. Younger patients present irritability, vomiting, and visual problems. Those older than three years usually show headaches, vomiting, and ataxia. [8]


-------------Worst Res Content------------
article-25161_14: Patients usually present within a month of treatment with the a